# Somatotopic Decoding — Linear SVC
**Task:** S1Map | **ROI:** Left S1 (Destrieux G_postcentral)  
**Classes:** Middle_Finger · Hand · Forearm · Arm (4-way, imbalanced)  
**Cross-validation:** Leave-One-Run-Out (LORO, 4 folds)  
**Classifier:** LinearSVC (C=3.0, OVR, `class_weight='balanced'`)  
**Features:** ROI BOLD averaged in a ±1-volume window around the HRF peak (~6 s delay)


In [ ]:
# Core imports and global random seed
import numpy as np

RANDOM_SEED = 42
rng = np.random.default_rng(RANDOM_SEED)


In [ ]:
# Electrode-to-region mapping (20 electrodes → 4 body regions)
REGION_TO_ELECTRODES = {
    'Middle_Finger': ['E1', 'E2', 'E3'],
    'Hand':          ['E4', 'E5', 'E6', 'E7'],
    'Forearm':       ['E8', 'E9', 'E10', 'E11', 'E12', 'E13'],
    'Arm':           ['E14', 'E15', 'E16', 'E17', 'E18', 'E19', 'E20'],
}

ELECTRODE_TO_REGION = {
    electrode: region
    for region, electrodes in REGION_TO_ELECTRODES.items()
    for electrode in electrodes
}


In [ ]:
# BIDS dataset paths and experiment parameters
from pathlib import Path
import json

BIDS_ROOT = Path(r"../../data/BIDS-somatosensory/BIDS-somatosensory")
DERIVATIVES = BIDS_ROOT / "derivatives" / "fmriprep"
EVENTS_DIR = BIDS_ROOT / "events"

session            = "ses-01"
task               = "task-S1Map"
space              = "MNI152NLin2009cAsym"
n_runs_per_subject = 4

HRF_DELAY = 6.0  # seconds from stimulus onset to HRF peak
WINDOW    = 1    # volumes to average on each side of the peak

In [ ]:
import re

derivative_subjects = sorted(d.name for d in DERIVATIVES.iterdir() if d.is_dir() and d.name.startswith("sub-"))

event_subjects = set()
for ev_path in EVENTS_DIR.glob(f"sub-*_{session}_{task}_run-*_events.tsv"):
    m = re.match(r"^(sub-[^_]+)_", ev_path.name)
    if m:
        event_subjects.add(m.group(1))
event_subjects = sorted(event_subjects)

subjects = sorted(set(derivative_subjects) & set(event_subjects))

print(f"Subjects in derivatives: {len(derivative_subjects)}")
print(f"Subjects in events:      {len(event_subjects)}")
print(f"Subjects included:       {len(subjects)}")
print(subjects)

In [ ]:
# get TR
def get_tr(subject, run=1, default_tr=2.0):
    func_dir = DERIVATIVES / subject / session / "func"

    json_candidates = [
        func_dir / f"{subject}_{session}_{task}_run-{run}_space-{space}_desc-preproc_bold.json",
        func_dir / f"{subject}_{session}_{task}_run-{run}_desc-preproc_bold.json",
    ]

    for bold_json in json_candidates:
        if bold_json.exists():
            with open(bold_json, "r", encoding="utf-8") as f:
                meta = json.load(f)

            if "RepetitionTime" in meta and meta["RepetitionTime"] is not None:
                return float(meta["RepetitionTime"])

            print(
                f"Warning: RepetitionTime missing in {bold_json.name} for {subject}. "
                f"Using default TR={default_tr} s."
            )
            return float(default_tr)

    print(
        f"Warning: No TR JSON found for {subject} (run {run}). "
        f"Using default TR={default_tr} s."
    )
    return float(default_tr)


tr_by_subject = {sub: get_tr(sub, run=1, default_tr=2.0) for sub in subjects}

unique_trs = sorted(set(tr_by_subject.values()))
tr = unique_trs[0]

print(f"Subjects kept: {len(subjects)}")
print(f"TR values across subjects: {unique_trs}")
if len(unique_trs) > 1:
    print("WARNING: TR is not identical across subjects; per-subject TR will be used.")
else:
    print(f"TR is consistent: {tr} s")
print(f"HRF delay: {HRF_DELAY} s  |  Window: +/-{WINDOW} vol")

In [ ]:
import pandas as pd

def load_subject_events(subject):
    all_events = []
    required_cols = {"onset", "duration", "trial_type"}

    for run in range(1, n_runs_per_subject + 1):
        events_path = EVENTS_DIR / f"{subject}_{session}_{task}_run-{run}_events.tsv"
        if not events_path.exists():
            raise FileNotFoundError(f"Missing events file: {events_path}")

        df = pd.read_csv(events_path, sep="\t")

        # Make headers consistent across files: Onset/Trial_Type -> onset/trial_type
        df.columns = [str(c).strip().lower() for c in df.columns]

        missing = required_cols - set(df.columns)
        if missing:
            raise ValueError(
                f"{events_path.name} missing columns {sorted(missing)}. "
                f"Found {list(df.columns)}"
            )

        df["subject"] = subject
        df["run"] = run
        all_events.append(df)

    events_df = pd.concat(all_events, ignore_index=True)

    stim_events = events_df[~events_df["trial_type"].isin(["Baseline", "Jitter"])].copy()
    stim_events["region"] = stim_events["trial_type"].map(ELECTRODE_TO_REGION)

    unmapped = int(stim_events["region"].isna().sum())
    if unmapped > 0:
        bad_labels = sorted(stim_events.loc[stim_events["region"].isna(), "trial_type"].unique())
        raise ValueError(f"Unmapped trial_type values for {subject}: {bad_labels}")

    return events_df, stim_events

# preview first subject
_events_df_preview, _stim_events_preview = load_subject_events(subjects[0])
print(f"Preview subject: {subjects[0]}")
print(f"Total events: {len(_events_df_preview)}  |  Stimulation events: {len(_stim_events_preview)}")
print(f"Samples per class: {_stim_events_preview['region'].value_counts().sort_index().to_dict()}")

In [ ]:
from nilearn.datasets import fetch_atlas_destrieux_2009
from nilearn.image import load_img, index_img, resample_to_img, new_img_like

atlas = fetch_atlas_destrieux_2009()
atlas_img = load_img(atlas.maps)
s1_indices = [
    i for i, lab in enumerate(atlas.labels)
    if "L G_postcentral" in str(lab) and i != 0
]
mask_data = np.isin(atlas_img.get_fdata(), s1_indices).astype("uint8")
s1_mask = new_img_like(atlas_img, mask_data)

In [ ]:
from nilearn.image import load_img, index_img, mean_img, resample_to_img
from nilearn.maskers import NiftiMasker
from nilearn import plotting
from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.metrics import accuracy_score, balanced_accuracy_score, confusion_matrix
import matplotlib.pyplot as plt
import numpy as np

RESULTS_DIR = Path("results")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

TOP_PERCENTILE = 50
summary_rows = []

for subject in subjects:
    print("\n" + "=" * 70)
    print(f"Processing {subject}")

    subject_results_dir = RESULTS_DIR / subject
    subject_figures_dir = subject_results_dir / "figures"
    subject_figures_dir.mkdir(parents=True, exist_ok=True)

    tr_subject = tr_by_subject[subject]
    events_df, stim_events = load_subject_events(subject)

    first_run_path = (
        DERIVATIVES / subject / session / "func"
        / f"{subject}_{session}_{task}_run-1_space-{space}_desc-preproc_bold.nii"
    )
    if not first_run_path.exists():
        print(f"Skipping {subject}: missing first run BOLD -> {first_run_path}")
        continue

    first_run_img = load_img(str(first_run_path))
    ref_img = index_img(first_run_img, 0)

    s1_mask_resampled = resample_to_img(s1_mask, ref_img, interpolation="nearest")
    masker = NiftiMasker(mask_img=s1_mask_resampled, standardize=None)
    masker.fit(first_run_img)

    feature_rows, labels, groups = [], [], []

    for run in range(1, n_runs_per_subject + 1):
        func_path = (
            DERIVATIVES / subject / session / "func"
            / f"{subject}_{session}_{task}_run-{run}_space-{space}_desc-preproc_bold.nii"
        )
        if not func_path.exists():
            print(f"  Missing BOLD run {run} for {subject}, skipping run.")
            continue

        img = load_img(str(func_path))
        run_length = img.shape[3]

        run_events = stim_events[
            (stim_events["subject"] == subject) &
            (stim_events["run"] == run)
        ].sort_values("onset")

        for _, ev in run_events.iterrows():
            peak_vol = int(np.round((float(ev["onset"]) + HRF_DELAY) / tr_subject))
            if peak_vol >= run_length:
                continue

            vols = list(range(max(0, peak_vol - WINDOW), min(run_length, peak_vol + WINDOW + 1)))
            averaged = mean_img(index_img(img, vols), copy_header=True)

            feature_rows.append(masker.transform(averaged).ravel())
            labels.append(str(ev["region"]))
            groups.append(run)

    if len(feature_rows) == 0:
        print(f"Skipping {subject}: no usable samples after extraction.")
        continue

    X_features = np.vstack(feature_rows)
    y = np.asarray(labels)
    groups = np.asarray(groups)

    unique_classes = np.unique(y)
    chance_level = 1.0 / len(unique_classes)

    print(f"Feature matrix: {X_features.shape} (samples x voxels)")
    print(f"Classes: {list(unique_classes)}")
    print(f"Samples/class: { {c: int(np.sum(y == c)) for c in unique_classes} }")

    logo = LeaveOneGroupOut()
    n_folds = logo.get_n_splits(groups=groups)
    if n_folds < 2:
        print(f"Skipping {subject}: not enough run groups for LORO.")
        continue

    clf = LinearSVC(
        C=3.0,
        class_weight="balanced",
        dual="auto",
        max_iter=20000,
        random_state=RANDOM_SEED,
    )
    pipe = Pipeline(steps=[("scale", StandardScaler()), ("clf", clf)])

    y_pred = np.empty_like(y, dtype=object)
    fold_acc, fold_bacc = [], []

    #store the values for each run for violin plots
    left_out_runs = []

    for fold_i, (train_idx, test_idx) in enumerate(logo.split(X_features, y, groups), 1):
        left_out = int(np.unique(groups[test_idx])[0])
        left_out_runs.append(left_out)

        pipe.fit(X_features[train_idx], y[train_idx])
        y_pred[test_idx] = pipe.predict(X_features[test_idx])

        acc = accuracy_score(y[test_idx], y_pred[test_idx])
        bacc = balanced_accuracy_score(y[test_idx], y_pred[test_idx])

        fold_acc.append(acc)
        fold_bacc.append(bacc)

        n_correct = int(np.sum(y[test_idx] == y_pred[test_idx]))
        n_total = len(test_idx)

        print(f"  Fold {fold_i} (left-out run {left_out}): {n_correct}/{n_total} correct")
        print(f"    Accuracy: {acc:.4f}   Balanced accuracy: {bacc:.4f}")

    fold_acc = np.asarray(fold_acc)
    fold_bacc = np.asarray(fold_bacc)

    fold_balanced_accuracy_df = pd.DataFrame(
        {
        "subject":subject,
        "fold":np.arange(1,len(fold_bacc)+1),
        "left_out_run": left_out_runs,
        "balanced_accuracy": fold_bacc,
        }
    )
    fold_balanced_accuracy_df.to_csv(
    subject_results_dir / "fold_balanced_accuracy.csv",
    index=False
    )

    mean_acc = float(fold_acc.mean())
    std_acc = float(fold_acc.std())
    mean_bacc = float(fold_bacc.mean())
    std_bacc = float(fold_bacc.std())

    print(f"  Mean accuracy:          {mean_acc:.4f} ± {std_acc:.4f}")
    print(f"  Mean balanced accuracy: {mean_bacc:.4f} ± {std_bacc:.4f}")
    print(f"  Chance level:           {chance_level:.4f}")

    fig, ax = plt.subplots(figsize=(10, 5))
    x = np.arange(1, len(fold_acc) + 1)
    width = 0.38

    ax.bar(
        x - width / 2,
        fold_acc,
        width=width,
        color="#1f77b4",
        alpha=0.85,
        edgecolor="black",
        label="Accuracy",
    )
    ax.bar(
        x + width / 2,
        fold_bacc,
        width=width,
        color="#ff7f0e",
        alpha=0.85,
        edgecolor="black",
        label="Balanced accuracy",
    )
    ax.axhline(
        chance_level,
        linestyle=":",
        linewidth=2,
        color="gray",
        label=f"Chance = {chance_level:.3f}",
    )
    ax.axhline(
        mean_bacc,
        linestyle="--",
        linewidth=2,
        color="red",
        alpha=0.5,
        label=f"Mean bacc = {mean_bacc:.3f}",
    )

    ax.set_xlabel("Fold (left-out run)", fontsize=12, fontweight="bold")
    ax.set_ylabel("Score", fontsize=12, fontweight="bold")
    ax.set_title(f"4-Way Region Classification ({subject})", fontsize=14, fontweight="bold")
    ax.set_xticks(x)
    ax.set_ylim(0, max(0.40, float(np.max(fold_bacc)) * 1.15))
    ax.grid(axis="y", alpha=0.3)
    ax.legend(loc="upper right")
    plt.tight_layout()
    fig.savefig(subject_figures_dir / "4way_performance.png", dpi=300, bbox_inches="tight")
    plt.close(fig)

    cm = confusion_matrix(y, y_pred, labels=unique_classes)
    row_sum = cm.sum(axis=1, keepdims=True)
    cm_norm = np.divide(cm, np.where(row_sum == 0, 1, row_sum))
    per_class_acc = np.diag(cm_norm)

    fig, axes = plt.subplots(1, 2, figsize=(14, 6))

    # Left: raw counts
    im0 = axes[0].imshow(cm, cmap="Blues", aspect="auto")
    axes[0].set_title("Confusion Matrix (Counts)", fontweight="bold")
    fig.colorbar(im0, ax=axes[0], fraction=0.046, pad=0.04)

    # Right: row-normalized
    im1 = axes[1].imshow(cm_norm, cmap="Blues", aspect="auto", vmin=0, vmax=1)
    axes[1].set_title("Confusion Matrix (Normalized)", fontweight="bold")
    fig.colorbar(im1, ax=axes[1], fraction=0.046, pad=0.04)

    # Write COUNT values on the counts matrix
    count_threshold = cm.max() / 2.0 if cm.size else 0
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            val = int(cm[i, j])
            color = "white" if val > count_threshold else "black"
            axes[0].text(j, i, f"{val}", ha="center", va="center", fontsize=8, color=color)

    # Write normalized values on the normalized matrix
    for i in range(cm_norm.shape[0]):
        for j in range(cm_norm.shape[1]):
            val = float(cm_norm[i, j])
            color = "white" if val > 0.5 else "black"
            axes[1].text(j, i, f"{val:.2f}", ha="center", va="center", fontsize=8, color=color)

    for ax in axes:
        ax.set_xticks(np.arange(len(unique_classes)))
        ax.set_yticks(np.arange(len(unique_classes)))
        ax.set_xticklabels(unique_classes, rotation=90)
        ax.set_yticklabels(unique_classes)
        ax.set_xlabel("Predicted Region")
        ax.set_ylabel("True Region")

    plt.tight_layout()
    fig.savefig(subject_figures_dir / "4way_confusion_matrix.png", dpi=300, bbox_inches="tight")
    plt.close(fig)

    full_pipe = Pipeline(steps=[("scale", StandardScaler()), ("clf", clone(clf))])
    full_pipe.fit(X_features, y)

    scaler = full_pipe.named_steps["scale"]
    svm = full_pipe.named_steps["clf"]
    class_labels = svm.classes_
    weight_voxel = svm.coef_ / scaler.scale_[np.newaxis, :]

    print(f"Weight shape: {weight_voxel.shape}  (n_classes={len(class_labels)}, n_voxels)")

    for class_index, class_name in enumerate(class_labels):
        abs_class_weights = np.abs(weight_voxel[class_index])
        threshold_value = np.percentile(abs_class_weights, 75)
        weight_img = masker.inverse_transform(abs_class_weights.reshape(1, -1))

        stat_display = plotting.plot_stat_map(
            weight_img,
            bg_img=ref_img,
            title=f"SVM Weights: {class_name}",
            threshold=threshold_value,
            colorbar=True,
            display_mode="ortho",
            cut_coords=(0, -20, 60),
        )
        stat_display.savefig(
            subject_figures_dir / f"svm_weights_{class_name}.png",
            dpi=300,
        )
        stat_display.close()

    abs_weights = np.abs(weight_voxel)
    winner_map = np.argmax(abs_weights, axis=0)
    max_weight = np.amax(abs_weights, axis=0)
    threshold_value = np.percentile(max_weight, TOP_PERCENTILE)
    winner_values = np.where(max_weight >= threshold_value, winner_map + 1, 0).astype(np.float64)
    winner_img = masker.inverse_transform(winner_values.reshape(1, -1))

    somatotopic_title = (
        "Somatotopic Map ("
        + ", ".join(f"{i + 1}={class_labels[i]}" for i in range(len(class_labels)))
        + ")"
    )

    winner_display = plotting.plot_roi(
        winner_img,
        bg_img=ref_img,
        title=somatotopic_title,
        display_mode="ortho",
        cut_coords=(0, -20, 60),
        cmap="tab10",
    )
    winner_display.savefig(
        subject_figures_dir / "somatotopic_map.png",
        dpi=300,
    )
    winner_display.close()

    surf_view = plotting.view_img_on_surf(
        winner_img,
        surf_mesh="fsaverage",
        title=somatotopic_title,
        symmetric_cmap=False,
        vmin=1,
        vmax=len(class_labels),
        threshold=1,
        vol_to_surf_kwargs={"interpolation": "nearest_most_frequent"},
    )
    surf_view.save_as_html(str(subject_figures_dir / "somatotopic_map_surface.html"))

    summary_rows.append(
        {
            "subject": subject,
            "n_samples": int(len(y)),
            "n_voxels": int(X_features.shape[1]),
            "n_classes": int(len(unique_classes)),
            "mean_accuracy": mean_acc,
            "std_accuracy": std_acc,
            "mean_balanced_accuracy": mean_bacc,
            "std_balanced_accuracy": std_bacc,
            "chance_level": chance_level,
            "tr": tr_subject,
            "samples_per_class": str({c: int(np.sum(y == c)) for c in unique_classes}),
            "mean_per_class_recall": float(np.mean(per_class_acc)),
        }
    )

summary_df = pd.DataFrame(summary_rows).sort_values("subject").reset_index(drop=True)
summary_csv = RESULTS_DIR / "svc_multi_subject_summary.csv"
summary_df.to_csv(summary_csv, index=False)

print("\n" + "=" * 70)
print(f"Finished subjects: {len(summary_df)}")
print(f"Summary saved: {summary_csv}")

if not summary_df.empty:
    global_mean = float(summary_df["mean_balanced_accuracy"].mean())
    global_std = float(summary_df["mean_balanced_accuracy"].std(ddof=0))
    print(f"Global mean balanced accuracy: {global_mean:.4f} ± {global_std:.4f}")

summary_df

In [ ]:
if "summary_df" not in globals() or summary_df.empty:
    print("summary_df is empty. Run the multi-subject pipeline cell first.")
else:
    fig, ax = plt.subplots(figsize=(12, 5))
    x = np.arange(len(summary_df))
    yvals = summary_df["mean_balanced_accuracy"].to_numpy()
    chance = float(summary_df["chance_level"].iloc[0])

    ax.bar(x, yvals, color="#2a9d8f", edgecolor="black", alpha=0.9)
    ax.axhline(chance, linestyle=":", linewidth=2, color="gray", label=f"Chance = {chance:.3f}")
    ax.axhline(np.mean(yvals), linestyle="--", linewidth=2, color="crimson",
               label=f"Mean = {np.mean(yvals):.3f}")

    ax.set_xticks(x)
    ax.set_xticklabels(summary_df["subject"], rotation=90)
    ax.set_ylabel("Mean Balanced Accuracy")
    ax.set_xlabel("Subject")
    ax.set_title("Per-Subject SVC Performance (LORO)", fontweight="bold")
    ax.set_ylim(0, max(0.5, float(np.max(yvals)) * 1.2))
    ax.grid(axis="y", alpha=0.3)
    ax.legend(loc="upper right")
    plt.tight_layout()

    out_path = RESULTS_DIR / "multi_subject_balanced_accuracy.png"
    fig.savefig(out_path, dpi=300, bbox_inches="tight")
    plt.show()
    print(f"Saved: {out_path}")
